STAGE 2: DATA CLEANING

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv("dataset.csv")

# First 5 rows
df.head()
# Last 5 rows
df.tail()
# Shape
df.shape
# Column names
df.columns
# Data types
df.dtypes
# General information
df.info()
'''
df describe give min/max/mean for every numerical column  
'''
df.describe()

3. Missing values
df.isnull().sum()
This gives you a count per column. Ask yourself: if a column has missing values, how many, and does it matter? A handful of missing track_name might be droppable; thousands missing in danceability would be a bigger problem.

In [ ]:
df.isnull().sum()
df[df["artists"].isnull()] #this will show you the row who has artist null
df=df.dropna(subset=["artists"]) #drop the row which has no value for artists

In [ ]:
''' 
when all columns were printed there was a column named 'unnamed' which basically was
classic pandas artifact-pandas default index included as extra column
'''
df['Unnamed: 0'].head()
df = df.drop(columns=['Unnamed: 0'])


In [ ]:
df.duplicated().sum() # fully identical rows outputs 450 we should remove these duplicates
df=df.drop_duplicates()
df.duplicated(subset=['track_name','artists']).sum()   #same song, different row (common in this dataset — same track appears in multiple genres) outputs 32206
df[df.duplicated(subset=['track_name','artists'], keep=False)].sort_values('track_name').head(10)
''' 
there are some songs which are listed twice because of the genre they have more than 1 genre so i keep all these songs and decide to treat "song+genre" 
as actual unit of analysis not just song
'''

In [ ]:
''' 
Each should return an empty dataframe if the data's 
 clean. If not, you've found something to fix or drop.
'''
df[df["danceability"]>1]
df[df["energy"]>1]
df[df["popularity"]>100]
df[df['tempo'] <= 0].shape[0]
#since there are only 157 rows who has tempo 0 means they have no rythm
df = df[df['tempo'] > 0]
#  Drop track_id — it's just an ID, not useful for analysis
df = df.drop(columns=['track_id'])


In [ ]:
# Final check after cleaning data
print(df.shape)
df.isnull().sum()

STAGE 2:  Exploratory Data Analysis (EDA)
Goal: Discover interesting patterns.

-> General:
What is the average popularity?
What is the median popularity?
Which song is the most popular?
Which song is the least popular?
What is the popularity distribution?

In [ ]:
#average popularity
average_popularity=df["popularity"].mean()
#median popularity
median_popularity=df["popularity"].median()
print(average_popularity,median_popularity)
#there are 2 methods for finding most popular song
#option:1
''' 
ascending=True → Lowest marks first.
ascending=False → Highest marks first.
.head(1) means: Give me the first 1 row.
'''
df.sort_values("popularity",ascending=False).head(1)
#option#2
df.loc[df['popularity'].idxmax()]
#least popular song:
df.loc[df["popularity"].idxmin()]
#popularity distribution:
print(df["popularity"].describe()) #summary stastics
plt.hist(df['popularity'],bins=50)
plt.title("Distribution of Song Popularity")
plt.xlabel("Popularity")
plt.ylabel("Number of Songs")
plt.show()

-> Genre Analysis
Which genre has the highest average popularity?
Which genre has the lowest?
Which genre has the highest average energy?
Which genre is the happiest (highest valence)?
Which genre is the most danceable?

In [ ]:
#high avg popularity acc to genre
grp=df.groupby("track_genre")
grp['popularity'].mean().sort_values(ascending=False).head(1)
#df.groupby("track_genre")["popularity"].mean().sort_values(ascending=False).head(1)
#low avg popularity acc to genre
grp['popularity'].mean().sort_values(ascending=True).head(1)
#high avg energy
grp['energy'].mean().sort_values(ascending=False).head(1)
#happiest genre
grp['valence'].mean().sort_values(ascending=False).head(1)
#most danceable
grp['danceability'].mean().sort_values(ascending=False).head(1)



-> Feature Relationships
Does energy correlate with popularity?
Does danceability correlate with popularity?
Does loudness correlate with popularity?
Does acousticness affect popularity?
Does tempo affect popularity?

What is Correlation?
Correlation tells us:
How strongly two variables are related to each other.
**positive correlation:**
For example, suppose we have students.
| Student | Hours Studied | Marks |
| ------- | ------------: | ----: |
| A       |             1 |    45 |
| B       |             2 |    55 |
| C       |             3 |    65 |
| D       |             4 |    75 |
| E       |             5 |    90 |
Notice:
As hours studied increase, marks also increase.
These two variables are positively correlated.
Marks
100 |                 •
 90 |             •
 80 |          •
 70 |       •
 60 |    •
 50 | •
    +------------------------
      1  2  3  4  5
      Hours Studied
**negative correlation:**
| Hours Playing Games | Exam Marks |
| ------------------: | ---------: |
|                   1 |         95 |
|                   2 |         88 |
|                   3 |         80 |
|                   4 |         72 |
|                   5 |         60 |
As gaming hours increase,
Marks decrease.
This is negative correlation.
Marks
100 | •
 90 |    •
 80 |       •
 70 |          •
 60 |             •
 50 |                •
    +------------------------
      1  2  3  4  5
      Hours Gaming
**no correlation:**
Suppose
| Shoe Size |  IQ |
| --------: | --: |
|         6 | 100 |
|         7 | 110 |
|         8 |  90 |
|         9 | 105 |

There is no meaningful relationship.
Knowing someone's shoe size doesn't help predict their IQ.
This is no correlation.
Marks
100 |      •
 90 |  •
 80 |          •
 70 |      •
 60 | •
 50 |            •
    +------------------------

**Correlation Coefficient**
The correlation coefficient is just a number that tells us:
How strong the relationship is.
Whether it's positive or negative.
It is usually represented by r.
The value of r is always between:
-1  ------------------  0  ------------------  +1
1. r = +1 : Perfect positive correlation.
when Hours Studied ↑ Marks ↑
Every increase in study hours leads to a proportional increase in marks.
2. r = -1 : Perfect negative correlation.
Gaming ↑ Marks ↓
3. r = 0: No relationship.
No clear pattern.
**strength of correlation**
| Correlation Coefficient | Meaning              |
| ----------------------: | -------------------- |
|                    +1.0 | Perfect positive     |
|                    +0.8 | Very strong positive |
|                    +0.5 | Moderate positive    |
|                    +0.2 | Weak positive        |
|                       0 | No correlation       |
|                    -0.2 | Weak negative        |
|                    -0.5 | Moderate negative    |
|                    -0.8 | Very strong negative |
|                    -1.0 | Perfect negative     |


In [ ]:
#correlation coefficient
print(df['popularity'].corr(df['energy']))
print(df['popularity'].corr(df['danceability']))
print(df['popularity'].corr(df['loudness']))
print(df['popularity'].corr(df['acousticness']))
print(df['popularity'].corr(df['tempo']))

In [ ]:
#visual way- scatterplot
#energy and popularity
sns.scatterplot(x='energy',y='popularity',data=df,alpha=0.3)


In [ ]:
#danceability and popularity
sns.scatterplot(x='danceability',y='popularity',data=df,alpha=0.3)

In [ ]:
#loudness and popularity
sns.scatterplot(x='loudness',y='popularity',data=df,alpha=0.3)


In [ ]:
#acousticness and popularity
sns.scatterplot(x='acousticness',y='popularity',data=df,alpha=0.3)

In [ ]:
#tempo and popularity
sns.scatterplot(x='tempo',y='popularity',data=df,alpha=0.3)

**Artists**
Which artists appear most frequently?
Which artist has the highest average popularity?

In [ ]:
#frequently appearing
df['artists'].value_counts().head(10)

In [ ]:
#high avg popularity
fap=df.groupby('artists')
fap['popularity'].mean().sort_values(ascending=False).head(10)

**Explicit Songs**
Are explicit songs generally more popular?

In [ ]:
es=df.groupby('explicit')
print(es['popularity'].mean())
print(df['explicit'].value_counts())
sns.boxplot(x='explicit', y='popularity', data=df)
print(es['popularity'].median())

In [ ]:
#High energy but low popularity:
df[(df['energy'] > 0.8) & (df['popularity'] < 20)][['track_name','artists','energy','popularity']].head(10)
#Very acoustic but still popular (the reverse case from your plan):
df[(df['acousticness'] > 0.8) & (df['popularity'] > 70)][['track_name','artists','acousticness','popularity']].head(10)
# High danceability but unpopular
df[(df['danceability'] > 0.8) & (df['popularity'] < 20)][['track_name','artists','danceability','popularity']].head(10)

# Very loud but unpopular
df[(df['loudness'] > -3) & (df['popularity'] < 20)][['track_name','artists','loudness','popularity']].head(10)

# High valence (happy-sounding) but very unpopular
df[(df['valence'] > 0.8) & (df['popularity'] < 10)][['track_name','artists','valence','popularity']].head(10)

**Create these plots**
Popularity histogram ✅
Popularity boxplot✅
Genre frequency bar chart✅
Top 10 genres by average popularity ✅
Top 10 artists by number of songs✅
Energy distribution ✅
Danceability distribution ✅
Tempo distribution✅
Scatter plot: Energy vs Popularity ✅
Scatter plot: Danceability vs Popularity✅
Scatter plot: Valence vs Popularity✅
Correlation heatmap
Pairplot of important numerical features
Boxplot of popularity by genre✅
Countplot of explicit vs non-explicit songs

In [ ]:
'''
value_counts() is essentially a shortcut that does "group
 by this column, count rows in each group, sort descending" 
'''
#Popularity histogram
plt.hist(df['popularity'],bins=50,color='lightgreen',edgecolor='black')
plt.title("popularity histogram")
plt.xlabel("popularity")
plt.ylabel("# of songs")
plt.grid(True)
plt.show()
#energy dist
plt.hist(df['energy'],bins=50,color='lightgreen',edgecolor='black')
plt.title("energy histogram")
plt.xlabel("energy")
plt.ylabel("# of songs")
plt.grid(True)
plt.show()
#danceability distribution 
plt.hist(df['danceability'],bins=50,color='lightgreen',edgecolor='black')
plt.title("danceability histogram")
plt.xlabel("danceability")
plt.ylabel("# of songs")
plt.grid(True)
plt.show()
#tempo distribution
plt.hist(df['tempo'],bins=50,color='lightgreen',edgecolor='black')
plt.title("tempo histogram")
plt.xlabel("tempo")
plt.ylabel("# of songs")
plt.grid(True)
plt.show()
#Genre frequency bar chart
genre_counts = df['track_genre'].value_counts().tail(10)
plt.bar(genre_counts.index, genre_counts.values, color='skyblue')
plt.title("Genre frequency bar chart")
plt.xlabel("track_genre")
plt.ylabel("# of songs")
plt.xticks(rotation=45)
plt.show()

#Top 10 genres by average popularity 
df.groupby("track_genre")['popularity'].mean().sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top 10 Genres by Avg Popularity')
plt.ylabel('Average Popularity')
plt.xticks(rotation=45)
plt.show()

#Top 10 artists by number of songs
df['artists'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Artists by Number of Songs')
plt.xticks(rotation=45)
plt.show()
#Scatter plot: Energy vs Popularity 
sns.scatterplot(x='energy',y='popularity',data=df,alpha=0.3)
plt.show()
#Scatter plot: Danceability vs Popularity
sns.scatterplot(x='danceability',y='popularity',data=df,alpha=0.3)
plt.show()
#Scatter plot: Valence vs Popularity
sns.scatterplot(x='valence',y='popularity',data=df,alpha=0.3)
plt.show()
#Popularity boxplot
sns.boxplot(y=df['popularity'])
plt.title('popularity boxplot')
plt.show()
# Boxplot of popularity BY genre - only makes sense for top genres, or it's unreadable
top_genres = df['track_genre'].value_counts().head(10).index
sns.boxplot(x='track_genre', y='popularity', data=df[df['track_genre'].isin(top_genres)])
plt.xticks(rotation=45)
plt.show()
#correlation heatmap
corr = df[['popularity','danceability','energy','loudness','acousticness','valence','tempo']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()
#pairplot
sample = df.sample(2000)  # IMPORTANT: pairplot on 114k rows will take forever/crash — sample first
sns.pairplot(sample[['popularity','energy','danceability','valence']])
plt.show()

